# Multicollinearity Motivation

**Project question:** Why can individually plausible OLS coefficients become unstable when predictors move together?

By the end of this notebook, you should be able to:

- identify correlated predictor groups and high variance-inflation factors
- separate coefficient instability from overall predictive signal
- explain why shrinkage may help prediction without proving causality

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
df = pd.read_csv(DATA / 'simulated_correlated_predictors.csv')
X = df.drop(columns=['id', 'weekly_sales'])
y = df['weekly_sales']
corr = X.corr()
plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title('Predictor correlation heatmap')
plt.tight_layout()

In [ ]:
X_const = sm.add_constant(X)
ols = sm.OLS(y, X_const).fit()
ols.summary()

In [ ]:
vif = pd.DataFrame({
    'variable': X_const.columns,
    'VIF': [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])],
})
vif.query("variable != 'const'").sort_values('VIF', ascending=False).head(10)

VIF quantifies how much the sampling variance of an OLS coefficient is inflated by linear association with the other predictors. A high VIF is not proof that a variable should be deleted; it signals that one-at-a-time coefficient interpretation may be fragile.

In [ ]:
rng = np.random.default_rng(4031)
bootstrap_coefs = []
for _ in range(100):
    sample_index = rng.integers(0, len(df), len(df))
    sample = df.iloc[sample_index]
    fitted = sm.OLS(
        sample['weekly_sales'],
        sm.add_constant(sample[X.columns]),
    ).fit()
    bootstrap_coefs.append(fitted.params.drop('const'))
coef_stability = pd.DataFrame(bootstrap_coefs).std().sort_values(ascending=False).to_frame('bootstrap_sd')
coef_stability.head(10)

**Interpretation:** Large bootstrap variation shows that correlated predictors can exchange credit across samples even when the combined predictive signal remains useful. Ridge addresses variance through shrinkage; it does not make the coefficients causal.